# R1-04 n8n POC (UC1 email triage + draft)

Spec: `specs/R1-04_n8n_poc.md`. Creates `n8n/airtable_seed.csv` and `n8n/workflow_documentation.md`.

`n8n/workflow.json` is frozen (prompt v5, tested in R1-05). This notebook only reads it and proves it is unchanged.

Run with Restart & Run All. Synthetic data only.

In [1]:
# 1 Check: right folder, on main, input files exist. Save BEFORE snapshots.
from pathlib import Path
import subprocess, shutil, datetime, hashlib, json, re
import pandas as pd

def git(*args):
    # small helper: run a read-only git command and return its output
    return subprocess.run(['git', *args], capture_output=True, text=True).stdout

REPO = Path(git('rev-parse', '--show-toplevel').strip())
N8N = REPO / 'n8n'
EMAILS = REPO / 'data' / 'processed' / 'spreekorb_emails_60.csv'
WORKFLOW = N8N / 'workflow.json'
NOTE = REPO / 'langsmith' / 'monitoring_note.md'
SEED = N8N / 'airtable_seed.csv'
DOC = N8N / 'workflow_documentation.md'
OUTPUTS = [SEED, DOC]

RUN_UNDO = False   # set True only to undo this card (cell 7)

def sha256(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def check(name, ok):
    ok = bool(ok)
    print(('PASS' if ok else 'FAIL'), '-', name)
    return ok

check('repo root found with git rev-parse', (REPO / 'AGENTS.md').exists())
branch = git('branch', '--show-current').strip()
check(f'on branch main (now: {branch})', branch == 'main')
inputs = [REPO / 'specs' / 'R1-04_n8n_poc.md', WORKFLOW, N8N / 'triage_prompt.md',
          N8N / 'airtable_setup.md', EMAILS, NOTE, REPO / 'langsmith' / 'trace_summary.csv']
for p in inputs:
    check(f'input exists: {p.relative_to(REPO)}', p.exists())

# BEFORE: git status (all untracked files listed) and hashes of files this card must not change
BEFORE = git('status', '--porcelain', '-uall').splitlines()
WF_SHA_BEFORE = sha256(WORKFLOW)
PROTECTED = [N8N / 'workflow.json', N8N / 'triage_prompt.md', N8N / 'airtable_setup.md']
for folder in ['knowledgebase', 'langsmith', 'data/processed', 'dashboard', 'specs']:
    PROTECTED += sorted(p for p in (REPO / folder).rglob('*') if p.is_file() and p.name != '.DS_Store')
PROTECTED_BEFORE = {str(p.relative_to(REPO)): sha256(p) for p in PROTECTED}
print(f'BEFORE: {len(BEFORE)} git status lines, {len(PROTECTED_BEFORE)} protected files hashed')
print('workflow.json sha256:', WF_SHA_BEFORE)

PASS - repo root found with git rev-parse
PASS - on branch main (now: main)
PASS - input exists: specs/R1-04_n8n_poc.md
PASS - input exists: n8n/workflow.json
PASS - input exists: n8n/triage_prompt.md
PASS - input exists: n8n/airtable_setup.md
PASS - input exists: data/processed/spreekorb_emails_60.csv
PASS - input exists: langsmith/monitoring_note.md
PASS - input exists: langsmith/trace_summary.csv
BEFORE: 8 git status lines, 48 protected files hashed
workflow.json sha256: 5777d1f92e9bb21ebb9492504537d6209c4e7431368f9f443734ec24f87207ba


In [2]:
# 2 Backup: copy output files that already exist to _backup/<timestamp>_R1-04/
STAMP = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
BACKUP = REPO / '_backup' / f'{STAMP}_R1-04'
existing = [p for p in OUTPUTS if p.exists()]
if existing:
    BACKUP.mkdir(parents=True, exist_ok=True)
    for p in existing:
        shutil.copy2(p, BACKUP / p.name)
        print('backed up:', p.relative_to(REPO))
    print('PASS - backup in', BACKUP.relative_to(REPO))
else:
    print('PASS - nothing to back up (no output file exists yet)')

PASS - nothing to back up (no output file exists yet)


In [3]:
# 3 Create n8n/airtable_seed.csv: 10 tuning emails (10 topics, 5 de + 5 en) + 1 empty email
emails = pd.read_csv(EMAILS, keep_default_na=False)
tuning = emails[emails['split'] == 'tuning'].sample(frac=1, random_state=42)   # shuffle, fixed seed

picked, used_topics, per_lang = [], set(), {'de': 0, 'en': 0}
for _, row in tuning.iterrows():
    # one email per topic, at most 5 per language
    if row['topic'] in used_topics or per_lang.get(row['language'], 5) >= 5:
        continue
    picked.append(row)
    used_topics.add(row['topic'])
    per_lang[row['language']] += 1
    if len(picked) == 10:
        break

seed = pd.DataFrame(picked)[['email_id', 'subject', 'body', 'language']].sort_values('email_id')
seed['status'] = 'New'
# test row for the "Mark as skipped" path: empty body
empty_row = pd.DataFrame([{'email_id': 'SEED-EMPTY', 'subject': 'Test: empty email (synthetic)',
                           'body': '', 'language': '', 'status': 'New'}])
seed = pd.concat([seed, empty_row], ignore_index=True).fillna('')
seed.to_csv(SEED, index=False, encoding='utf-8')
print(f'written {SEED.relative_to(REPO)}: {len(seed)} rows')
print(seed[['email_id', 'language', 'status']].assign(
    topic=seed['email_id'].map(emails.set_index('email_id')['topic'])).to_string(index=False))

written n8n/airtable_seed.csv: 11 rows
  email_id language status                             topic
       S08       en    New Late delivery, no tracking update
       S14       en    New      Cancellation before dispatch
       S19       de    New                    Return request
       S23       de    New              Exchange, wrong size
       S25       de    New               Wrong item received
       S28       en    New                    Refund overdue
       S37       de    New                    Invoice resend
       S38       en    New                     Double charge
       S56       en    New                      Legal threat
       S59       de    New                 Thanks / feedback
SEED-EMPTY             New                               NaN


In [4]:
# 4 Create n8n/workflow_documentation.md
# Held-out figures are copied from langsmith/monitoring_note.md (not recomputed)
note = NOTE.read_text(encoding='utf-8')

def heldout(evaluator):
    # row "| evaluator | what | dev | held-out |" -> held-out value
    m = re.search(rf'^\| {evaluator} \|[^\n]*\| ([^|\n]+) \|\s*$', note, re.M)
    return m.group(1).strip() if m else 'MISSING'

EVALS = ['queue_match', 'priority_match', 'priority_within_one', 'no_banned_phrase',
         'escalation_flagged', 'language_match', 'word_count', 'signoff_ok', 'no_unknown_status']
FIG = {e: heldout(e) for e in EVALS}
m = re.search(r'experiment (heldout-v5-[0-9a-f]+)', note)
EXPERIMENT = m.group(1) if m else 'MISSING'
m = re.search(r'Time and cost \(heldout_20\): ([^\n]*?total [0-9.]+ USD)', note)
TIME_COST = m.group(1) if m else 'MISSING'
fig_lines = '\n'.join(f'| {e} | {v} |' for e, v in FIG.items())

DOC_TEXT = f"""# n8n workflow documentation: R1-04 (UC1 email triage + draft)

Card R1-04, spec `specs/R1-04_n8n_poc.md`. Workflow: `n8n/workflow.json` (8 nodes, inactive on import). Written by `notebooks/R1-04_n8n_poc.ipynb`. Synthetic emails only.

## Purpose
UC1 for Spreekorb email customer service. For each new email the AI suggests a queue and a priority, gives a one-line reason and writes a reply draft. A person reads every draft and decides: approve, edit or reject. No email is sent: the workflow has no email node, it only writes back to the Airtable table Review queue.

## Nodes
The spec asks for 3 steps: trigger, AI triage + draft, output for human review. The workflow keeps 8 nodes (Ayhan's decision). The extra nodes are small helpers, and each one belongs to one of the 3 steps. The model and the output parser are sub-nodes of Triage + draft, not extra steps.

| # | Node (n8n type) | What it does | Why it fits | Spec step |
|---|---|---|---|---|
| 1 | New email in Review queue (Airtable Trigger) | Checks the Review queue table every minute for rows with a new `created` time | Airtable is the inbox for the POC, so no real mailbox is needed | trigger |
| 2 | Clean email (Edit Fields) | Keeps record_id, email_id, subject and body, trims spaces | Only the fields the AI needs are passed on (data minimisation) | trigger |
| 3 | Has email text? (If) | Body not empty: go to the AI. Empty: go to Mark as skipped | No AI cost and no invented draft for an empty email | AI triage + draft (guard) |
| 4 | Triage + draft (Basic LLM Chain) | Sends subject and body with system prompt v5 (`n8n/triage_prompt.md` plus the knowledge base) | Same prompt as tested in LangSmith (R1-05); cell 5 checks the node text against `triage_prompt.md` | AI triage + draft |
| 5 | OpenAI gpt-4o-mini (sub-node of 4) | The model, temperature 0.2 | Low cost per draft (see R1-05), low temperature for steady output | AI triage + draft |
| 6 | Output format (Structured Output Parser, sub-node of 4) | Forces JSON: language, queue, priority, reason, draft_reply, policy_sections | Only allowed queue and priority values reach Airtable | AI triage + draft |
| 7 | Save draft for review (Airtable update) | Writes the AI fields, status `Pending review`, model `gpt-4o-mini, prompt v5`, processed_at | The person sees draft, reason and cited policy sections side by side | output for human review |
| 8 | Mark as skipped (Airtable update) | Sets status `Skipped: empty email` | An empty email stays visible to a person instead of vanishing | output for human review |

## Setup
1. Airtable: create base Spreekorb Email Triage and table Review queue with the 16 fields in `n8n/airtable_setup.md`.
2. n8n: Workflows > Import from file > `n8n/workflow.json`.
3. Credentials in n8n: add an Airtable Personal Access Token credential and an OpenAI credential. Select the Airtable one in the 3 Airtable nodes (the trigger and both update nodes) and the OpenAI one in the model node. Keys stay in n8n, never in this repo.
4. In the 3 Airtable nodes replace `PASTE_BASE_ID` (app...) and `PASTE_TABLE_ID` (tbl...) with the ids from the Airtable URL. Refresh the column list in both update nodes.

## How to run
1. Activate the workflow first, then import `n8n/airtable_seed.csv` into Review queue (11 synthetic rows, all status `New`: 10 tuning emails with 10 topics, 5 de and 5 en, plus SEED-EMPTY). The trigger only picks up rows created after activation (`airtable_setup.md`, step 6), so within about a minute it fetches the new rows. For a single check you can also use Test workflow.
2. Check in Airtable: the 10 email rows change to status `Pending review`, with ai_language, ai_queue, ai_priority, ai_reason, draft_reply and policy_sections filled.
3. Check that SEED-EMPTY becomes `Skipped: empty email` and gets no draft.
4. Human review: set 3 rows to Approved, Edited or Rejected and fill final_reply or reviewer_note. Nothing is sent. reviewer_note and final_reply must not contain agent names, agent IDs or other personal data.
5. Deactivate the workflow after the test (the trigger polls every minute).

## Link to evaluation
Prompt v5 in the Triage + draft node is the version tested in LangSmith (R1-05). Held-out set spreekorb-heldout-20 (20 synthetic emails, never used for tuning, human labels by Ayhan), experiment {EXPERIMENT}. Figures copied from `langsmith/monitoring_note.md`, not recomputed:

| Evaluator (held-out, share passing) | Result |
|---|---|
{fig_lines}

Time and cost: {TIME_COST} (LangSmith, measured). Code checks test form, not truth, so a person still reviews every draft. Only 20 emails: this shows the method works, it does not prove quality yet.

## Limits vs production
- Synthetic emails only, no real Spreekorb customers.
- Polling trigger (every minute) on an Airtable table, not a live mailbox or webhook.
- No error workflow, retry or alert yet (Round 2).
- Live n8n runs are not traced in LangSmith; only the R1-05 notebook runs are (Round 2).
- A person reviews every draft. No email is sent automatically.
- Processors: OpenAI (model), Airtable (table) and n8n (workflow and execution logs; EU cloud region or self-hosted). Before any real customer data, each needs a data processing agreement, a transfer basis (SCCs or DPF) and an EU region where offered. Set n8n execution data pruning. OpenAI API data retention and no-training terms are to be confirmed in the DPA. The works agreement is still a draft (`langsmith/monitoring_note.md`). This POC is prepared for compliance, not proof of it.
- EU AI Act (our assessment, not legal advice): a drafting assistant where a person reviews and sends every reply is likely minimal or limited risk, with no direct AI-to-customer contact. Automatic sending would need a new transparency review. Staff who use the tool need AI literacy (Art. 4).
- Works council: no agent names or IDs in Airtable (no Assignee or Last modified by fields). Approval and edit rates are reported at team level only, minimum group of 5.

## Screenshots
Placeholders, Ayhan adds the files:
- Workflow in n8n: `n8n/screenshots/workflow.png`

  ![n8n workflow](screenshots/workflow.png)
- Airtable review view: `n8n/screenshots/airtable_review.png`

  ![Airtable review queue](screenshots/airtable_review.png)
"""
DOC.write_text(DOC_TEXT, encoding='utf-8')
print(f'written {DOC.relative_to(REPO)}: {len(DOC_TEXT.split())} words')
print('held-out figures copied:', FIG, EXPERIMENT)

written n8n/workflow_documentation.md: 1090 words
held-out figures copied: {'queue_match': '85%', 'priority_match': '70%', 'priority_within_one': '95%', 'no_banned_phrase': '100%', 'escalation_flagged': '100%', 'language_match': '100%', 'word_count': '100%', 'signoff_ok': '100%', 'no_unknown_status': '100%'} heldout-v5-489260bd


In [5]:
# 5 Verify: each acceptance item prints PASS or FAIL
results = []
s = pd.read_csv(SEED, keep_default_na=False).fillna('').astype(str)
filled = s[s['body'].str.strip() != '']
src = emails.set_index('email_id')
results.append(check('seed has 11 rows', len(s) == 11))
results.append(check('seed columns are email_id, subject, body, language, status',
                     list(s.columns) == ['email_id', 'subject', 'body', 'language', 'status']))
topics = filled['email_id'].map(src['topic'])
results.append(check('10 unique topics', topics.nunique() == 10 and len(filled) == 10))
lang = filled['language'].value_counts().to_dict()
results.append(check(f'5 de / 5 en (now {lang})', lang == {'de': 5, 'en': 5}))
empty = s[s['body'].str.strip() == '']
results.append(check('1 empty body row, email_id SEED-EMPTY',
                     len(empty) == 1 and empty['email_id'].iloc[0] == 'SEED-EMPTY'))
results.append(check('all status = New', (s['status'] == 'New').all()))
results.append(check('all non-empty rows are tuning',
                     filled['email_id'].isin(src.index).all()
                     and (filled['email_id'].map(src['split']) == 'tuning').all()))
results.append(check('seed text equals source (subject, body, language)',
                     all((filled[c].to_numpy() == filled['email_id'].map(src[c]).to_numpy()).all()
                         for c in ['subject', 'body', 'language'])))

doc = DOC.read_text(encoding='utf-8')
SECTIONS = ['## Purpose', '## Nodes', '## Setup', '## How to run', '## Link to evaluation',
            '## Limits vs production', '## Screenshots']
missing = [h for h in SECTIONS if h not in doc]
results.append(check(f'documentation has all sections (missing: {missing})', not missing))
results.append(check('documentation has no em dash', chr(0x2014) not in doc))
wf = json.loads(WORKFLOW.read_text(encoding='utf-8'))
names = [n['name'] for n in wf['nodes']]
results.append(check('documentation names all 8 workflow nodes',
                     len(names) == 8 and all(n in doc for n in names)))
results.append(check('held-out figures copied from monitoring_note.md (none missing)',
                     'MISSING' not in FIG.values() and EXPERIMENT != 'MISSING' and TIME_COST != 'MISSING'))
node_rows = [l for l in doc.splitlines() if re.match(r'^\| [1-8] \|', l)]
rows_full = len(node_rows) == 8 and all(all(c.strip() for c in l.strip().strip('|').split('|')) for l in node_rows)
results.append(check('docs: node table has 8 full rows (what it does, why it fits, spec step)',
                     '| What it does | Why it fits |' in doc and rows_full))
lim = doc.split('## Limits vs production')[1].split('## Screenshots')[0] if '## Limits vs production' in doc else ''
results.append(check('docs: limits vs production name processors, DPA, AI Act and works council',
                     all(w in lim for w in ['Processors', 'data processing agreement', 'AI Act', 'Works council', 'minimum group of 5'])))
results.append(check('docs state that no email is sent', 'No email is sent' in doc))
seed_text = ' '.join(s['subject'] + ' ' + s['body'])
pii = re.findall(r'@|\bDE\d{2}[ ]?\d{4}|\+?\d[\d /-]{8,}\d', seed_text)
results.append(check(f'seed has no email address, IBAN or phone-like number (found: {pii})', not pii))

results.append(check('workflow.json sha256 unchanged', sha256(WORKFLOW) == WF_SHA_BEFORE))
types = [n['type'].lower() for n in wf['nodes']]
send_words = ['email', 'gmail', 'smtp', 'outlook', 'sendgrid', 'mailgun', 'mailchimp', 'microsoftoutlook']
send_nodes = [t for t in types if any(w in t for w in send_words)]
results.append(check(f'workflow.json has no send-email node (found: {send_nodes})', not send_nodes))
at_ops = [n['parameters'].get('operation') for n in wf['nodes'] if n['type'] == 'n8n-nodes-base.airtable']
results.append(check(f'Airtable nodes only update rows (ops: {at_ops})', at_ops and set(at_ops) == {'update'}))
results.append(check('workflow.json active is false', wf.get('active') is False))
node_prompt = next(n for n in wf['nodes'] if n['name'] == 'Triage + draft')['parameters']['messages']['messageValues'][0]['message']
system_v5 = (N8N / 'triage_prompt.md').read_text(encoding='utf-8').split('## SYSTEM', 1)[1].split('{knowledge_base}')[0].strip()
results.append(check('Triage + draft node uses the triage_prompt.md v5 SYSTEM text', system_v5 in node_prompt))
print('NOTE - spec "3-4 nodes": workflow has 8 nodes by Ayhan\'s decision, mapped to the 3 spec steps in the doc (not a PASS)')
print('MANUAL - runs on sample emails: open until Ayhan runs it in n8n and adds the 2 screenshots (not counted below)')
print(f'\n{sum(results)}/{len(results)} PASS')

PASS - seed has 11 rows
PASS - seed columns are email_id, subject, body, language, status
PASS - 10 unique topics
PASS - 5 de / 5 en (now {'en': 5, 'de': 5})
PASS - 1 empty body row, email_id SEED-EMPTY
PASS - all status = New
PASS - all non-empty rows are tuning
PASS - seed text equals source (subject, body, language)
PASS - documentation has all sections (missing: [])
PASS - documentation has no em dash
PASS - documentation names all 8 workflow nodes
PASS - held-out figures copied from monitoring_note.md (none missing)
PASS - docs: node table has 8 full rows (what it does, why it fits, spec step)
PASS - docs: limits vs production name processors, DPA, AI Act and works council
PASS - docs state that no email is sent
PASS - seed has no email address, IBAN or phone-like number (found: [])
PASS - workflow.json sha256 unchanged
PASS - workflow.json has no send-email node (found: [])
PASS - Airtable nodes only update rows (ops: ['update', 'update'])
PASS - workflow.json active is false
PAS

In [6]:
# 6 Unchanged proof: git status vs BEFORE, and hashes of protected files
AFTER = git('status', '--porcelain', '-uall').splitlines()
ALLOWED = {str(p.relative_to(REPO)) for p in OUTPUTS}
changed = sorted(set(AFTER) ^ set(BEFORE))
unexpected = [line for line in changed if line[3:].strip('"') not in ALLOWED]
print('git status lines that changed:', changed or 'none')
check(f'only the new files differ in git status (unexpected: {unexpected})', not unexpected)
PROTECTED_AFTER = {k: sha256(REPO / k) if (REPO / k).exists() else 'deleted' for k in PROTECTED_BEFORE}
diff = [k for k in PROTECTED_BEFORE if PROTECTED_AFTER[k] != PROTECTED_BEFORE[k]]
check(f'{len(PROTECTED_BEFORE)} protected files unchanged (changed: {diff})', not diff)

git status lines that changed: ['?? n8n/airtable_seed.csv', '?? n8n/workflow_documentation.md']
PASS - only the new files differ in git status (unexpected: [])
PASS - 48 protected files unchanged (changed: [])


True

In [ ]:
# 7 Undo: remove the created files and restore any backup (runs only if RUN_UNDO = True)
if RUN_UNDO:
    for p in OUTPUTS:
        if p.exists():
            p.unlink()
            print('removed:', p.relative_to(REPO))
    if BACKUP.exists():
        for f in BACKUP.iterdir():
            shutil.copy2(f, N8N / f.name)
            print('restored:', (N8N / f.name).relative_to(REPO))
    check('outputs removed or restored from backup',
          all(p.exists() == (BACKUP / p.name).exists() for p in OUTPUTS))
else:
    print('Undo skipped (RUN_UNDO = False)')

In [7]:
# 8 Git: print the commands. Never run them here. Ayhan commits.
print('git add n8n/airtable_seed.csv n8n/workflow_documentation.md notebooks/R1-04_n8n_poc.ipynb board.md logs/agent_log.md')
print('git commit -m "R1-04 n8n POC: Airtable seed CSV and workflow documentation"')

git add n8n/airtable_seed.csv n8n/workflow_documentation.md notebooks/R1-04_n8n_poc.ipynb board.md logs/agent_log.md
git commit -m "R1-04 n8n POC: Airtable seed CSV and workflow documentation"
